# **LoFTR Reproduction: Dataset Preparation and Relative-Pose Evaluation**


Sethmini Denegama

This notebook contains only :
- official ScanNet-1500 and MegaDepth-1500 test-data preparation,
- benchmark-specific preprocessing,
- pretrained LoFTR inference needed for evaluation,
- relative-pose estimation and pose-error calculation,
- AUC evaluation at 5°, 10°, and 20°,
- saving the reproduced quantitative results.




In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json
from pathlib import Path
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from copy import deepcopy
from tqdm.auto import tqdm

REPO_ROOT = Path("/content/LoFTR")
if not REPO_ROOT.exists():
    raise FileNotFoundError(
        "Expected /content/LoFTR. Run the group's repository/environment setup first."
    )

os.chdir(REPO_ROOT)
sys.path.append(str(REPO_ROOT))

from src.loftr import LoFTR, default_cfg

print("Working directory:", os.getcwd())

In [ ]:
SCANNET_TAR = "/content/drive/MyDrive/LoFTR/testdata/Copy of scannet_test_1500.tar"
SCANNET_ROOT = "/content/LoFTR/data/scannet/test"
SCANNET_TEST_NPZ = "/content/LoFTR/assets/scannet_test_1500/test.npz"
SCANNET_INTR_NPZ = "/content/LoFTR/assets/scannet_test_1500/intrinsics.npz"

os.makedirs(SCANNET_ROOT, exist_ok=True)

# Extract once
if not os.path.exists(os.path.join(SCANNET_ROOT, "scene0707_00")):
    !tar -xf "$SCANNET_TAR" -C "$SCANNET_ROOT" --strip-components=1

test_data = np.load(SCANNET_TEST_NPZ)
pairs = test_data["name"]
intr_data = np.load(SCANNET_INTR_NPZ)

print("Official ScanNet pairs:", len(pairs))
print("Pair array shape:", pairs.shape)
print("First 3 pairs:")
print(pairs[:3])

In [ ]:
# Pretrained indoor dual-softmax checkpoint
cfg_indoor = deepcopy(default_cfg)
cfg_indoor["coarse"]["temp_bug_fix"] = True

matcher_indoor = LoFTR(config=cfg_indoor)

INDOOR_CKPT = "/content/drive/MyDrive/LoFTR/weights/Copy of indoor_ds_new.ckpt"
checkpoint = torch.load(INDOOR_CKPT, map_location="cpu")

matcher_indoor.load_state_dict(checkpoint["state_dict"])
matcher_indoor = matcher_indoor.eval().cuda()

print("Indoor LoFTR loaded successfully.")

## ScanNet-1500

### One ScanNet verification pair

In [ ]:
scene = "scene0772_00"
frame0, frame1 = 1020, 1050

img0_path = os.path.join(SCANNET_ROOT, scene, "color", f"{frame0}.jpg")
img1_path = os.path.join(SCANNET_ROOT, scene, "color", f"{frame1}.jpg")

img0_raw = cv2.imread(img0_path, cv2.IMREAD_GRAYSCALE)
img1_raw = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)

img0_raw = cv2.resize(img0_raw, (640, 480))
img1_raw = cv2.resize(img1_raw, (640, 480))

img0 = torch.from_numpy(img0_raw).float()[None, None].cuda() / 255.0
img1 = torch.from_numpy(img1_raw).float()[None, None].cuda() / 255.0

batch = {"image0": img0, "image1": img1}

with torch.no_grad():
    matcher_indoor(batch)

mkpts0 = batch["mkpts0_f"].detach().cpu().numpy()
mkpts1 = batch["mkpts1_f"].detach().cpu().numpy()
mconf = batch["mconf"].detach().cpu().numpy()

print("Number of LoFTR matches:", len(mkpts0))

In [ ]:
# Plot the 150 highest-confidence correspondences
idx = np.argsort(mconf)[::-1][:150]
pts0_vis, pts1_vis = mkpts0[idx], mkpts1[idx]

h = max(img0_raw.shape[0], img1_raw.shape[0])
w0, w1 = img0_raw.shape[1], img1_raw.shape[1]

canvas = np.zeros((h, w0 + w1), dtype=np.uint8)
canvas[:img0_raw.shape[0], :w0] = img0_raw
canvas[:img1_raw.shape[0], w0:w0+w1] = img1_raw

plt.figure(figsize=(16, 8))
plt.imshow(canvas, cmap="gray")

for p0, p1 in zip(pts0_vis, pts1_vis):
    x0, y0 = p0
    x1, y1 = p1
    x1 += w0
    plt.plot([x0, x1], [y0, y1], linewidth=0.7, alpha=0.7)
    plt.scatter(x0, y0, s=5)
    plt.scatter(x1, y1, s=5)

plt.title(f"LoFTR: Top 150 Matches\nTotal predicted matches = {len(mkpts0)}")
plt.axis("off")
plt.show()

In [ ]:
# Relative-pose and AUC functions

def estimate_pose_scannet(kpts0, kpts1, K0, K1, pixel_thr=0.5, conf=0.99999):
    if len(kpts0) < 5:
        return None

    # Normalize coordinates by camera intrinsics
    p0 = (kpts0 - K0[[0, 1], [2, 2]]) / K0[[0, 1], [0, 1]]
    p1 = (kpts1 - K1[[0, 1], [2, 2]]) / K1[[0, 1], [0, 1]]

    focal = np.mean([K0[0, 0], K0[1, 1], K1[0, 0], K1[1, 1]])
    ransac_thr = pixel_thr / focal

    E, mask = cv2.findEssentialMat(
        p0, p1, np.eye(3),
        threshold=ransac_thr,
        prob=conf,
        method=cv2.RANSAC
    )

    if E is None:
        return None

    best_num_inliers = 0
    best_result = None

    for E_candidate in np.split(E, len(E) // 3):
        n, R, t, _ = cv2.recoverPose(
            E_candidate, p0, p1, np.eye(3), 1e9, mask=mask
        )

        if n > best_num_inliers:
            best_num_inliers = n
            best_result = (R, t[:, 0], mask.ravel() > 0)

    return best_result


def relative_pose_error_scannet(T_gt, R_est, t_est):
    R_gt = T_gt[:3, :3]
    t_gt = T_gt[:3, 3]

    denom = np.linalg.norm(t_est) * np.linalg.norm(t_gt)

    if denom == 0:
        t_err = 0.0
    else:
        cos_t = np.dot(t_est, t_gt) / denom
        cos_t = np.clip(cos_t, -1.0, 1.0)
        t_err = np.degrees(np.arccos(cos_t))
        # Essential-matrix translation has sign ambiguity
        t_err = min(t_err, 180.0 - t_err)

    cos_R = (np.trace(R_est.T @ R_gt) - 1.0) / 2.0
    cos_R = np.clip(cos_R, -1.0, 1.0)
    R_err = np.degrees(np.arccos(cos_R))

    return t_err, R_err


def error_auc(errors, thresholds=(5, 10, 20)):
    errors = [0.0] + sorted(list(errors))
    recall = np.linspace(0, 1, len(errors)).tolist()

    results = {}
    for thr in thresholds:
        last_index = np.searchsorted(errors, thr)
        y = recall[:last_index] + [recall[last_index - 1]]
        x = errors[:last_index] + [thr]
        results[f"AUC@{thr}°"] = np.trapezoid(y, x) / thr

    return results

### Full ScanNet-1500 evaluation

In [ ]:
pose_errors = []
rotation_errors = []
translation_errors = []
match_counts = []
inlier_counts = []
failed_pairs = []

matcher_indoor.eval()

for idx, pair in enumerate(tqdm(pairs, desc="ScanNet-1500")):
    scene_id, subscene_id, frame0, frame1 = map(int, pair)
    scene = f"scene{scene_id:04d}_{subscene_id:02d}"

    img0_path = os.path.join(SCANNET_ROOT, scene, "color", f"{frame0}.jpg")
    img1_path = os.path.join(SCANNET_ROOT, scene, "color", f"{frame1}.jpg")
    pose0_path = os.path.join(SCANNET_ROOT, scene, "pose", f"{frame0}.txt")
    pose1_path = os.path.join(SCANNET_ROOT, scene, "pose", f"{frame1}.txt")

    try:
        img0 = cv2.imread(img0_path, cv2.IMREAD_GRAYSCALE)
        img1 = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)

        if img0 is None or img1 is None:
            raise FileNotFoundError("Missing image")

        img0 = cv2.resize(img0, (640, 480))
        img1 = cv2.resize(img1, (640, 480))

        ten0 = torch.from_numpy(img0).float()[None, None].cuda() / 255.0
        ten1 = torch.from_numpy(img1).float()[None, None].cuda() / 255.0

        batch = {"image0": ten0, "image1": ten1}

        with torch.no_grad():
            matcher_indoor(batch)

        pts0 = batch["mkpts0_f"].detach().cpu().numpy()
        pts1 = batch["mkpts1_f"].detach().cpu().numpy()
        match_counts.append(len(pts0))

        K0 = intr_data[scene].copy()
        K1 = intr_data[scene].copy()

        result = estimate_pose_scannet(
            pts0, pts1, K0, K1,
            pixel_thr=0.5,
            conf=0.99999
        )

        if result is None:
            pose_errors.append(np.inf)
            rotation_errors.append(np.inf)
            translation_errors.append(np.inf)
            inlier_counts.append(0)
            failed_pairs.append(idx)
            continue

        R_est, t_est, inliers = result
        inlier_counts.append(int(inliers.sum()))

        cam2world0 = np.loadtxt(pose0_path)
        cam2world1 = np.loadtxt(pose1_path)
        T_gt = np.linalg.inv(cam2world1) @ cam2world0

        t_err, R_err = relative_pose_error_scannet(T_gt, R_est, t_est)

        translation_errors.append(t_err)
        rotation_errors.append(R_err)
        pose_errors.append(max(t_err, R_err))

    except Exception as e:
        pose_errors.append(np.inf)
        rotation_errors.append(np.inf)
        translation_errors.append(np.inf)
        match_counts.append(0)
        inlier_counts.append(0)
        failed_pairs.append(idx)
        print(f"Failed pair {idx}: {scene} {frame0}-{frame1}: {e}")

print("\nScanNet evaluation complete.")
print("Pairs evaluated:", len(pose_errors))
print("Failed pairs:", len(failed_pairs))

In [ ]:
scannet_aucs = error_auc(pose_errors)

finite_scannet = np.asarray(pose_errors, dtype=float)
finite_scannet = finite_scannet[np.isfinite(finite_scannet)]

print("===== ScanNet-1500 Reproduction =====")
for name, value in scannet_aucs.items():
    print(f"{name}: {value:.4f}  ({value * 100:.2f}%)")

print("\nMedian pose error:", np.median(finite_scannet))
print("Average matches per pair:", np.mean(match_counts))
print("Average RANSAC inliers per pair:", np.mean(inlier_counts))
print("Failed pairs:", len(failed_pairs))

In [ ]:
scannet_results = {
    "dataset": "ScanNet-1500",
    "model": "LoFTR indoor dual-softmax",
    "num_pairs": len(pose_errors),
    "failed_pairs": len(failed_pairs),
    "auc_5": float(scannet_aucs["AUC@5°"]),
    "auc_10": float(scannet_aucs["AUC@10°"]),
    "auc_20": float(scannet_aucs["AUC@20°"]),
    "median_pose_error_deg": float(np.median(finite_scannet)),
    "average_matches": float(np.mean(match_counts)),
    "average_ransac_inliers": float(np.mean(inlier_counts))
}

SCANNET_RESULTS_PATH = "/content/drive/MyDrive/LoFTR/scannet_reproduction_results.json"

with open(SCANNET_RESULTS_PATH, "w") as f:
    json.dump(scannet_results, f, indent=4)

print("Saved:", SCANNET_RESULTS_PATH)

## MegaDepth-1500

In [ ]:
MEGA_TAR = "/content/drive/MyDrive/LoFTR/testdata/Copy of megadepth_test_1500.tar"
MEGA_ROOT = "/content/LoFTR/data/megadepth/test"
SCENE_ROOT = "/content/LoFTR/assets/megadepth_test_1500_scene_info"

os.makedirs(MEGA_ROOT, exist_ok=True)

if not os.path.exists(os.path.join(MEGA_ROOT, "Undistorted_SfM")):
    !tar -xf "$MEGA_TAR" -C "$MEGA_ROOT" --strip-components=1

with open(os.path.join(SCENE_ROOT, "megadepth_test_1500.txt")) as f:
    scenes = [line.strip() for line in f if line.strip()]

print("MegaDepth subsets:", scenes)

In [ ]:
cfg_outdoor = deepcopy(default_cfg)

# Original outdoor_ds.ckpt uses the old positional encoding
cfg_outdoor["coarse"]["temp_bug_fix"] = False

matcher_outdoor = LoFTR(config=cfg_outdoor)

OUTDOOR_CKPT = "/content/drive/MyDrive/LoFTR/weights/Copy of outdoor_ds.ckpt"
checkpoint = torch.load(OUTDOOR_CKPT, map_location="cpu")

matcher_outdoor.load_state_dict(checkpoint["state_dict"])
matcher_outdoor = matcher_outdoor.eval().cuda()

print("Outdoor LoFTR loaded with temp_bug_fix = False")

In [ ]:
from src.datasets.megadepth import MegaDepthDataset

datasets = []

for scene_name in scenes:
    ds = MegaDepthDataset(
        root_dir=MEGA_ROOT,
        npz_path=os.path.join(SCENE_ROOT, scene_name + ".npz"),
        mode="test",
        min_overlap_score=0.0,
        img_resize=840,
        df=8,
        img_padding=True,
        depth_padding=False
    )

    datasets.append(ds)
    print(scene_name, "pairs:", len(ds))

print("\nTotal pairs:", sum(len(ds) for ds in datasets))

### Full MegaDepth-1500 evaluation

In [ ]:
from src.utils.metrics import (
    estimate_pose as estimate_pose_repo,
    relative_pose_error as relative_pose_error_repo
)

mega_pose_errors = []
mega_rotation_errors = []
mega_translation_errors = []
mega_match_counts = []
mega_inlier_counts = []
mega_failed_pairs = []

global_idx = 0
matcher_outdoor.eval()

for scene_name, ds in zip(scenes, datasets):
    for local_idx in tqdm(range(len(ds)), desc=scene_name):

        try:
            sample = ds[local_idx]

            batch = {
                "image0": sample["image0"][None].cuda(),
                "image1": sample["image1"][None].cuda(),
                "scale0": sample["scale0"][None].cuda(),
                "scale1": sample["scale1"][None].cuda(),
            }

            if "mask0" in sample:
                batch["mask0"] = sample["mask0"][None].cuda()
                batch["mask1"] = sample["mask1"][None].cuda()

            with torch.no_grad():
                matcher_outdoor(batch)

            pts0 = batch["mkpts0_f"].cpu().numpy()
            pts1 = batch["mkpts1_f"].cpu().numpy()
            mega_match_counts.append(len(pts0))

            K0 = sample["K0"].numpy()
            K1 = sample["K1"].numpy()
            T_gt = sample["T_0to1"].numpy()

            ret = estimate_pose_repo(
                pts0, pts1, K0, K1,
                thresh=0.5,
                conf=0.99999
            )

            if ret is None:
                mega_pose_errors.append(np.inf)
                mega_rotation_errors.append(np.inf)
                mega_translation_errors.append(np.inf)
                mega_inlier_counts.append(0)
                mega_failed_pairs.append(global_idx)

            else:
                R, t, inliers = ret
                mega_inlier_counts.append(int(inliers.sum()))

                t_err, R_err = relative_pose_error_repo(T_gt, R, t)

                mega_translation_errors.append(t_err)
                mega_rotation_errors.append(R_err)
                mega_pose_errors.append(max(t_err, R_err))

        except Exception as e:
            mega_pose_errors.append(np.inf)
            mega_rotation_errors.append(np.inf)
            mega_translation_errors.append(np.inf)
            mega_match_counts.append(0)
            mega_inlier_counts.append(0)
            mega_failed_pairs.append(global_idx)
            print(f"Failed pair {global_idx}: {e}")

        global_idx += 1

        # Recovery file in case the long benchmark run is interrupted
        if global_idx % 100 == 0:
            np.save(
                "/content/drive/MyDrive/LoFTR/megadepth_pose_errors_partial.npy",
                np.asarray(mega_pose_errors, dtype=float)
            )

print("\nMegaDepth evaluation complete.")
print("Pairs evaluated:", len(mega_pose_errors))
print("Failed pairs:", len(mega_failed_pairs))

In [ ]:
mega_aucs = error_auc(mega_pose_errors)

finite_mega = np.asarray(mega_pose_errors, dtype=float)
finite_mega = finite_mega[np.isfinite(finite_mega)]

print("===== MegaDepth-1500 Reproduction =====")
for name, value in mega_aucs.items():
    print(f"{name}: {value:.4f}  ({value * 100:.2f}%)")

print("\nMedian pose error:", np.median(finite_mega))
print("Average matches per pair:", np.mean(mega_match_counts))
print("Average RANSAC inliers per pair:", np.mean(mega_inlier_counts))
print("Failed pairs:", len(mega_failed_pairs))

In [ ]:
mega_results = {
    "dataset": "MegaDepth-1500",
    "model": "LoFTR outdoor dual-softmax",
    "num_pairs": len(mega_pose_errors),
    "failed_pairs": len(mega_failed_pairs),
    "auc_5": float(mega_aucs["AUC@5°"]),
    "auc_10": float(mega_aucs["AUC@10°"]),
    "auc_20": float(mega_aucs["AUC@20°"]),
    "median_pose_error_deg": float(np.median(finite_mega)),
    "average_matches": float(np.mean(mega_match_counts)),
    "average_ransac_inliers": float(np.mean(mega_inlier_counts))
}

MEGA_RESULTS_PATH = "/content/drive/MyDrive/LoFTR/megadepth_reproduction_results.json"
MEGA_ERRORS_PATH = "/content/drive/MyDrive/LoFTR/megadepth_pose_errors_final.npy"

with open(MEGA_RESULTS_PATH, "w") as f:
    json.dump(mega_results, f, indent=4)

np.save(MEGA_ERRORS_PATH, np.asarray(mega_pose_errors, dtype=float))

print("Saved:", MEGA_RESULTS_PATH)
print("Saved:", MEGA_ERRORS_PATH)

## Reproduced outputs

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "Dataset": ["ScanNet-1500", "MegaDepth-1500"],
    "Pairs": [len(pose_errors), len(mega_pose_errors)],
    "Failed pose estimates": [len(failed_pairs), len(mega_failed_pairs)],
    "AUC @ 5° (%)": [
        scannet_aucs["AUC@5°"] * 100,
        mega_aucs["AUC@5°"] * 100
    ],
    "AUC @ 10° (%)": [
        scannet_aucs["AUC@10°"] * 100,
        mega_aucs["AUC@10°"] * 100
    ],
    "AUC @ 20° (%)": [
        scannet_aucs["AUC@20°"] * 100,
        mega_aucs["AUC@20°"] * 100
    ],
    "Median pose error (°)": [
        np.median(finite_scannet),
        np.median(finite_mega)
    ]
})

print(summary.round(2).to_string(index=False))

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls -lh "/content/drive/MyDrive/Colab Notebooks" | grep "LoFTR_SETH_CLEAN_REVIEWED"

-rw------- 1 root root  22K Nov  8 05:42 LoFTR_SETH_CLEAN_REVIEWED.ipynb


In [4]:
!cp "/content/drive/MyDrive/Colab Notebooks/LoFTR_SETH_CLEAN_REVIEWED.ipynb" \
"/content/LoFTR/reproduction/Sethmini/LoFTR_SETH_CLEAN_REVIEWED.ipynb"

cp: cannot create regular file '/content/LoFTR/reproduction/Sethmini/LoFTR_SETH_CLEAN_REVIEWED.ipynb': No such file or directory


In [5]:
%cd /content/LoFTR
!git branch --show-current
!ls reproduction/Sethmini

[Errno 2] No such file or directory: '/content/LoFTR'
/content
fatal: not a git repository (or any of the parent directories): .git
ls: cannot access 'reproduction/Sethmini': No such file or directory


In [6]:
%cd /content
!git clone https://github.com/matthewpinto1102-web/LoFTR.git
%cd /content/LoFTR

/content
Cloning into 'LoFTR'...
remote: Enumerating objects: 425, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 425 (delta 134), reused 112 (delta 107), pack-reused 216 (from 1)
Receiving objects: 100% (425/425), 31.74 MiB | 34.54 MiB/s, done.
Resolving deltas: 100% (161/161), done.
/content/LoFTR


In [7]:
!git checkout -b Sethmini-evaluation
!mkdir -p reproduction/Sethmini

Switched to a new branch 'Sethmini-evaluation'


In [8]:
!cp "/content/drive/MyDrive/Colab Notebooks/LoFTR_SETH_CLEAN_REVIEWED.ipynb" \
"/content/LoFTR/reproduction/Sethmini/"

In [9]:
!cp "/content/drive/MyDrive/LoFTR/scannet_reproduction_results.json" \
"/content/LoFTR/reproduction/Sethmini/"

!cp "/content/drive/MyDrive/LoFTR/megadepth_reproduction_results.json" \
"/content/LoFTR/reproduction/Sethmini/"

In [10]:
!ls -lh reproduction/Sethmini

total 32K
-rw------- 1 root root 24K Nov  8 05:44 LoFTR_SETH_CLEAN_REVIEWED.ipynb
-rw------- 1 root root 330 Nov  8 05:44 megadepth_reproduction_results.json
-rw------- 1 root root 316 Nov  8 05:44 scannet_reproduction_results.json


In [11]:
%cd /content/LoFTR

!git config user.name "Sethmini Denegama"
!git config user.email "sdenegam@terpmail.umd.edu"

/content/LoFTR


In [12]:
!git add reproduction/Sethmini/
!git status

On branch Sethmini-evaluation
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   reproduction/Sethmini/LoFTR_SETH_CLEAN_REVIEWED.ipynb
	new file:   reproduction/Sethmini/megadepth_reproduction_results.json
	new file:   reproduction/Sethmini/scannet_reproduction_results.json



In [13]:
!git commit -m "Add ScanNet and MegaDepth evaluation reproduction"

[Sethmini-evaluation d5010b6] Add ScanNet and MegaDepth evaluation reproduction
 3 files changed, 25 insertions(+)
 create mode 100644 reproduction/Sethmini/LoFTR_SETH_CLEAN_REVIEWED.ipynb
 create mode 100644 reproduction/Sethmini/megadepth_reproduction_results.json
 create mode 100644 reproduction/Sethmini/scannet_reproduction_results.json


In [14]:
!git push -u origin Sethmini-evaluation

fatal: could not read Username for 'https://github.com': No such device or address


In [15]:
!git log -1 --oneline

d5010b6 (HEAD -> Sethmini-evaluation) Add ScanNet and MegaDepth evaluation reproduction


In [16]:
!gh auth login -h github.com -p https -w

! Did you know that expiring tokens can be requested with --expiring? Read more at https://gh.io/gh-expiring-tokens
? Authenticate Git with your GitHub credentials? (Y/n) 7^C
